In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('imdb-top-1000.csv')
movies.head()

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
0,The Shawshank Redemption,1994,142,Drama,9.3,Frank Darabont,Tim Robbins,2343110,28341469.0,80.0
1,The Godfather,1972,175,Crime,9.2,Francis Ford Coppola,Marlon Brando,1620367,134966411.0,100.0
2,The Dark Knight,2008,152,Action,9.0,Christopher Nolan,Christian Bale,2303232,534858444.0,84.0
3,The Godfather: Part II,1974,202,Crime,9.0,Francis Ford Coppola,Al Pacino,1129952,57300000.0,90.0
4,12 Angry Men,1957,96,Crime,9.0,Sidney Lumet,Henry Fonda,689845,4360000.0,96.0


# Ultimate Guide to Pandas `groupby`

The `groupby` operation in Pandas is one of the most powerful tools for data analysis. It follows the **Split-Apply-Combine** strategy introduced by Hadley Wickham. This guide covers everything from basic mechanics to advanced optimizations.

---

## 1. The Core Concept: Split-Apply-Combine

To understand `groupby`, you must understand its three distinct phases:

* **Split:** Breaking a DataFrame into groups based on one or more keys.
* **Apply:** Passing a function to each group independently.
* **Combine:** Merging the individual results back into a single DataFrame or Series.

> 💡 **Key Insight:** When you run `df.groupby('column')`, Pandas doesn't immediately compute anything. It creates a `DataFrameGroupBy` object, which is a lazy evaluation mechanism waiting for an **Apply** step.

---

## 2. Creating GroupBy Objects (The "Split" Phase)

You can group data in multiple ways depending on your architecture and goals.

### A. Single vs. Multiple Keys

* **Single Key:** `df.groupby('Region')`
* **Multiple Keys (Creates MultiIndex):** `df.groupby(['Region', 'Year'])`

### B. Grouping by Columns vs. Index/Level

If your dataframe already has an index or a MultiIndex, you can group by it directly:

* `df.groupby(level=0)` or `df.groupby(level='City')`

### C. Grouping by Expressions or Arrays

You can pass external arrays, series, or mappings (dictionaries/functions) to group data dynamically.

```python
# Grouping by a custom condition (e.g., Even vs Odd index)
df.groupby(df['Age'] > 30)

# Grouping via a dictionary mapping
mapping = {'Apple': 'Fruit', 'Broccoli': 'Veggie', 'Banana': 'Fruit'}
df.groupby(mapping)

# Grouping by string methods on the index
df.groupby(lambda x: x.upper())

```

---

## 3. Inspecting and Iterating over Groups

Before applying functions, you often need to debug or understand what groups were formed.

* **`groupby.groups`**: Returns a dictionary where keys are group names and values are the corresponding index labels.
* **`groupby.indices`**: Similar to `.groups`, but returns integer positions instead of labels.
* **`groupby.get_group('GroupName')`**: Extracts a single group as a standalone DataFrame.
* **Iterating through groups:**
```python
for name, group in df.groupby('Region'):
    print(f"Group: {name}")
    print(group.head(2))

```



---

## 4. The "Apply" Phase: Aggregation, Transformation, Filtration

This is where the real work happens. There are three main classes of operations you can perform on groups.

### A. Aggregation (`.agg()` or `.aggregate()`)

Aggregation reduces each group to a **single scalar value**.

* **Built-in Helpers:** `.sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.count()`, `.std()`, `.var()`.
* **Named Aggregation (Recommended):** Prevents messy column multi-indices.
```python
df.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Average_Rating=('Rating', 'mean')
)

```


* **Dictionary Aggregation:** Applying different functions to different columns.
```python
df.groupby('Region').agg({'Sales': 'sum', 'Age': ['min', 'max']})

```



### B. Transformation (`.transform()`)

Transformation returns an object that is the **exact same size** as the original input group. It is incredibly useful for broadcasting group-level statistics back to individual rows (e.g., calculating z-scores or filling missing values with group means).

```python
# Fill missing values with the group's mean
df['Sales'] = df.groupby('Region')['Sales'].transform(lambda x: x.fillna(x.mean()))

# Calculate percentage contribution of each row to its group total
df['Pct_of_Region'] = df['Sales'] / df.groupby('Region')['Sales'].transform('sum')

```

### C. Filtration (`.filter()`)

Filtration discards entire groups based on a boolean condition evaluated at the group level.

```python
# Keep only regions where the total sales exceed $1,000,000
df_filtered = df.groupby('Region').filter(lambda x: x['Sales'].sum() > 1000000)

```

### D. The Flexible `.apply()`

When your logic doesn't neatly fit into aggregation, transformation, or filtration, `.apply()` handles arbitrary Python functions. It passes each group as a whole DataFrame to your custom function.

```python
# Get the top 2 highest-earning rows for each group
def top_two(group):
    return group.sort_values(by='Sales', ascending=False).head(2)

df.groupby('Region').apply(top_two,純_index=False)

```

> ⚠️ **Performance Note:** `.apply()` is significantly slower than `.agg()` or `.transform()` because it cannot take advantage of Pandas' vectorized internal C-routines. Use it as a last resort.

---

## 5. Handling GroupBy Outputs & MultiIndex

Grouping by multiple columns produces a `MultiIndex` (hierarchical index) in the result. Managing this output is crucial for downstream analysis.

| Method / Parameter | Description | Example |
| --- | --- | --- |
| `as_index=False` | Keeps grouping keys as regular columns instead of setting them as the index. | `df.groupby('Reg', as_index=False).sum()` |
| `.reset_index()` | Converts a MultiIndex back into standard columns post-computation. | `df.groupby(['A', 'B']).mean().reset_index()` |
| `.unstack()` | Pivots one of the levels of a MultiIndex into columns. | `df.groupby(['Year', 'Reg'])['Sales'].sum().unstack()` |

---

## 6. Advanced GroupBy Mechanics & Edge Cases

### A. Missing Values in Grouping Keys

By default, Pandas drops rows where the grouping key is `NaN`. If you want to treat `NaN` as its own valid group, use the `dropna` parameter:

```python
df.groupby('Region', dropna=False).sum()

```

### B. Grouping by Time (Resampling within Groups)

Combining `groupby` with `pd.Grouper` allows you to perform complex time-series aggregations per group.

```python
# Group by Region, and then bucket Sales into Monthly intervals
df.groupby(['Region', pd.Grouper(key='DateColumn', freq='ME')])['Sales'].sum()

```

### C. The `observed` Parameter for Categoricals

If you group by a column that has a `Categorical` data type, Pandas will create groups for *all* possible categories—even those with zero records in the data. To prevent empty rows/groups:

```python
df.groupby('Categorical_Col', observed=True).sum()

```

---

## 7. Performance & Optimization Tips

When dealing with large datasets (millions of rows), unoptimized `groupby` statements can bottleneck your pipeline.

1. **Use Categoricals for Strings:** Grouping by string columns is slow. Convert string grouping columns to `'category'` types first; it speeds up hashing significantly.
```python
df['Region'] = df['Region'].astype('category')

```


2. **Avoid Lambdas if Built-ins Exist:** `groupby().transform(lambda x: x.sum())` is vastly slower than `groupby().transform('sum')`. The string shortcut utilizes optimized Cython routines.
3. **Filter Columns *Before* Grouping:** If you only need one column aggregated, specify it before the operation rather than calculating it for the whole DataFrame.
* *Bad:* `df.groupby('Reg').sum()['Sales']` (computes sum for all columns, throws away others)
* *Good:* `df.groupby('Reg')['Sales'].sum()` (only computes sum for 'Sales')


4. **`sort=False` for Speed:** By default, Pandas sorts the group keys. If you don't care about alphabetical/numerical order of the keys, turn it off to save processing time:
```python
df.groupby('Region', sort=False).sum()

```

In [3]:
genres = movies.groupby('Genre')
type(genres)

pandas.core.groupby.generic.DataFrameGroupBy

In [4]:
# Applying aggregation functions on groupby objects
genres.max()

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,Yôjinbô,2019,321,9.0,Zack Snyder,Yun-Fat Chow,2303232,936662225.0,98.0
Adventure,Zombieland,PG,228,8.6,Ömer Faruk Sorak,Yves Montand,1512360,874211619.0,100.0
Animation,Ôkami kodomo no Ame to Yuki,2020,137,8.6,Yoshifumi Kondô,Yôji Matsuda,999790,873839108.0,96.0
Biography,Zerkalo,2020,209,8.9,Tom McCarthy,Éric Toledano,1213505,753585104.0,97.0
Comedy,Zindagi Na Milegi Dobara,2020,188,8.6,Zoya Akhtar,Ömer Faruk Sorak,939631,886752933.0,99.0
Crime,À bout de souffle,2019,229,9.2,Yavuz Turgul,Vincent Cassel,1826188,790482117.0,100.0
Drama,Zwartboek,2020,242,9.3,Çagan Irmak,Çetin Tekindor,2343110,924558264.0,100.0
Family,Willy Wonka & the Chocolate Factory,1982,115,7.8,Steven Spielberg,Henry Thomas,372490,435110554.0,91.0
Fantasy,Nosferatu,1922,94,8.1,Robert Wiene,Werner Krauss,88794,445151978.0,NaN


In [5]:
# for name,group in movies.groupby('Genre'):
#     print(f'Group : {name}')
#     print(group.head(2))

print(movies.groupby('Genre').get_group('Action'))
print(movies.groupby('Genre').indices)
print(movies.groupby('Genre').groups)

                                          Series_Title Released_Year  Runtime  \
2                                      The Dark Knight          2008      152   
5        The Lord of the Rings: The Return of the King          2003      201   
8                                            Inception          2010      148   
10   The Lord of the Rings: The Fellowship of the Ring          2001      178   
13               The Lord of the Rings: The Two Towers          2002      179   
..                                                 ...           ...      ...   
968                                       Falling Down          1993      113   
979                                      Lethal Weapon          1987      109   
982                                          Mad Max 2          1981       96   
983                                       The Warriors          1979       92   
985                               Escape from Alcatraz          1979      112   

      Genre  IMDB_Rating   

In [6]:
movies.head(2)

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
0,The Shawshank Redemption,1994,142,Drama,9.3,Frank Darabont,Tim Robbins,2343110,28341469.0,80.0
1,The Godfather,1972,175,Crime,9.2,Francis Ford Coppola,Marlon Brando,1620367,134966411.0,100.0


In [7]:
movies.groupby('Genre').sum()['Gross'].sort_values(ascending=False)

Genre
Drama        3.540997e+10
Action       3.263226e+10
Comedy       1.566387e+10
Animation    1.463147e+10
Adventure    9.496922e+09
Crime        8.452632e+09
Biography    8.276358e+09
Mystery      1.256417e+09
Horror       1.034649e+09
Fantasy      7.827267e+08
Family       4.391106e+08
Film-Noir    1.259105e+08
Western      5.822151e+07
Thriller     1.755074e+07
Name: Gross, dtype: float64

In [8]:
movies.groupby('Genre')['Gross'].sum().sort_values(ascending=False)

Genre
Drama        3.540997e+10
Action       3.263226e+10
Comedy       1.566387e+10
Animation    1.463147e+10
Adventure    9.496922e+09
Crime        8.452632e+09
Biography    8.276358e+09
Mystery      1.256417e+09
Horror       1.034649e+09
Fantasy      7.827267e+08
Family       4.391106e+08
Film-Noir    1.259105e+08
Western      5.822151e+07
Thriller     1.755074e+07
Name: Gross, dtype: float64

In [9]:
movies.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False)

Genre
Western      8.350000
Crime        8.016822
Fantasy      8.000000
Mystery      7.975000
Film-Noir    7.966667
Drama        7.957439
Action       7.949419
Biography    7.938636
Adventure    7.937500
Animation    7.930488
Horror       7.909091
Comedy       7.901290
Family       7.800000
Thriller     7.800000
Name: IMDB_Rating, dtype: float64

In [10]:
movies.head(2)
movies.groupby('Director')['No_of_Votes'].sum().sort_values(ascending=False).head(1)

Director
Christopher Nolan    11578345
Name: No_of_Votes, dtype: int64

In [11]:
movies.head(2)
# find the highest rated movie of each genre
movies.groupby('Genre')['IMDB_Rating'].max().sort_values(ascending=False)

Genre
Drama        9.3
Crime        9.2
Action       9.0
Biography    8.9
Western      8.8
Adventure    8.6
Comedy       8.6
Animation    8.6
Horror       8.5
Mystery      8.4
Fantasy      8.1
Film-Noir    8.1
Family       7.8
Thriller     7.8
Name: IMDB_Rating, dtype: float64

In [12]:
# find number of movies done by each actor
movies.head(2)
movies.groupby('Star1')['Series_Title'].count().sort_values(ascending=False)

Star1
Tom Hanks               12
Robert De Niro          11
Clint Eastwood          10
Al Pacino               10
Humphrey Bogart          9
                        ..
Zbigniew Zamachowski     1
Zooey Deschanel          1
Çetin Tekindor           1
Éric Toledano            1
Aaron Taylor-Johnson     1
Name: Series_Title, Length: 660, dtype: int64

In [13]:
# GroupBy Attributes and methods
# Find Total num of groups -> len
# Find items in each group -> size
# first() / last() -> nth item
# get_group -> vs filtering
# groups
# describe
# sample
# nunique

In [14]:
len(movies.groupby('Genre'))

14

In [15]:
(movies.groupby('Genre').size())

Genre
Action       172
Adventure     72
Animation     82
Biography     88
Comedy       155
Crime        107
Drama        289
Family         2
Fantasy        2
Film-Noir      3
Horror        11
Mystery       12
Thriller       1
Western        4
dtype: int64

In [16]:
movies.groupby('Genre').first()

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,The Dark Knight,2008,152,9.0,Christopher Nolan,Christian Bale,2303232,534858444.0,84.0
Adventure,Interstellar,2014,169,8.6,Christopher Nolan,Matthew McConaughey,1512360,188020017.0,74.0
Animation,Sen to Chihiro no kamikakushi,2001,125,8.6,Hayao Miyazaki,Daveigh Chase,651376,10055859.0,96.0
Biography,Schindler's List,1993,195,8.9,Steven Spielberg,Liam Neeson,1213505,96898818.0,94.0
Comedy,Gisaengchung,2019,132,8.6,Bong Joon Ho,Kang-ho Song,552778,53367844.0,96.0
Crime,The Godfather,1972,175,9.2,Francis Ford Coppola,Marlon Brando,1620367,134966411.0,100.0
Drama,The Shawshank Redemption,1994,142,9.3,Frank Darabont,Tim Robbins,2343110,28341469.0,80.0
Family,E.T. the Extra-Terrestrial,1982,115,7.8,Steven Spielberg,Henry Thomas,372490,435110554.0,91.0
Fantasy,Das Cabinet des Dr. Caligari,1920,76,8.1,Robert Wiene,Werner Krauss,57428,337574718.0,NaN


In [28]:
movies['Genre'].nunique()

14

In [29]:
genres = movies.groupby('Genre')
genres.nth(6) # 6th ele,ent of every group in genre

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
16,Star Wars: Episode V - The Empire Strikes Back,1980,124,Action,8.7,Irvin Kershner,Mark Hamill,1159315,290475067.0,82.0
27,Se7en,1995,127,Crime,8.6,David Fincher,Morgan Freeman,1445096,100125643.0,65.0
32,It's a Wonderful Life,1946,130,Drama,8.6,Frank Capra,James Stewart,405801,82385199.0,89.0
66,WALL·E,2008,98,Animation,8.4,Andrew Stanton,Ben Burtt,999790,223808164.0,95.0
83,The Great Dictator,1940,125,Comedy,8.4,Charles Chaplin,Charles Chaplin,203150,288475.0,NaN
102,Braveheart,1995,178,Biography,8.3,Mel Gibson,Mel Gibson,959181,75600000.0,68.0
118,North by Northwest,1959,136,Adventure,8.3,Alfred Hitchcock,Cary Grant,299198,13275000.0,98.0
420,Sleuth,1972,138,Mystery,8.0,Joseph L. Mankiewicz,Laurence Olivier,44748,4081254.0,NaN
724,Get Out,2017,104,Horror,7.7,Jordan Peele,Daniel Kaluuya,492851,176040665.0,85.0


In [19]:
movies['Genre'].value_counts()

Genre
Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Fantasy        2
Family         2
Thriller       1
Name: count, dtype: int64

In [20]:
genres.get_group('Action')

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
2,The Dark Knight,2008,152,Action,9.0,Christopher Nolan,Christian Bale,2303232,534858444.0,84.0
5,The Lord of the Rings: The Return of the King,2003,201,Action,8.9,Peter Jackson,Elijah Wood,1642758,377845905.0,94.0
8,Inception,2010,148,Action,8.8,Christopher Nolan,Leonardo DiCaprio,2067042,292576195.0,74.0
10,The Lord of the Rings: The Fellowship of the Ring,2001,178,Action,8.8,Peter Jackson,Elijah Wood,1661481,315544750.0,92.0
13,The Lord of the Rings: The Two Towers,2002,179,Action,8.7,Peter Jackson,Elijah Wood,1485555,342551365.0,87.0
...,...,...,...,...,...,...,...,...,...,...
968,Falling Down,1993,113,Action,7.6,Joel Schumacher,Michael Douglas,171640,40903593.0,56.0
979,Lethal Weapon,1987,109,Action,7.6,Richard Donner,Mel Gibson,236894,65207127.0,68.0
982,Mad Max 2,1981,96,Action,7.6,George Miller,Mel Gibson,166588,12465371.0,77.0
983,The Warriors,1979,92,Action,7.6,Walter Hill,Michael Beck,93878,22490039.0,65.0


In [21]:
movies[movies['Genre']=='Fantasy']

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
321,Das Cabinet des Dr. Caligari,1920,76,Fantasy,8.1,Robert Wiene,Werner Krauss,57428,337574718.0,NaN
568,Nosferatu,1922,94,Fantasy,7.9,F.W. Murnau,Max Schreck,88794,445151978.0,NaN


In [22]:
genres.describe()

Runtime                                                              \
            count        mean        std    min     25%    50%     75%    max   
Genre                                                                           
Action      172.0  129.046512  28.500706   45.0  110.75  127.5  143.25  321.0   
Adventure    72.0  134.111111  33.317320   88.0  109.00  127.0  149.00  228.0   
Animation    82.0   99.585366  14.530471   71.0   90.00   99.5  106.75  137.0   
Biography    88.0  136.022727  25.514466   93.0  120.00  129.0  146.25  209.0   
Comedy      155.0  112.129032  22.946213   68.0   96.00  106.0  124.50  188.0   
Crime       107.0  126.392523  27.689231   80.0  106.50  122.0  141.50  229.0   
Drama       289.0  124.737024  27.740490   64.0  105.00  121.0  137.00  242.0   
Family        2.0  107.500000  10.606602  100.0  103.75  107.5  111.25  115.0   
Fantasy       2.0   85.000000  12.727922   76.0   80.50   85.0   89.50   94.0   
Film-Noir     3.0  104.000000   4.000000  100.0  102.00  104.0  106.00  108.0   
Horror       11.0  102.090909  13.604812   71.0   98.00  103.0  109.00  122.0   
Mystery      12.0  119.083333  14.475423   96.0  110.75  117.5  130.25  138.0   
Thriller      1.0  108.000000        NaN  108.0  108.00  108.0  108.00  108.0   
Western       4.0  148.250000  17.153717  132.0  134.25  148.0  162.00  165.0   

          IMDB_Rating            ...         Gross              Metascore  \
                count      mean  ...           75%          max     count   
Genre                            ...                                        
Action          172.0  7.949419  ...  2.674437e+08  936662225.0     143.0   
Adventure        72.0  7.937500  ...  1.998070e+08  874211619.0      64.0   
Animation        82.0  7.930488  ...  2.520612e+08  873839108.0      75.0   
Biography        88.0  7.938636  ...  9.829924e+07  753585104.0      79.0   
Comedy          155.0  7.901290  ...  8.107809e+07  886752933.0     125.0   
Crime           107.0  8.016822  ...  7.102163e+07  790482117.0      87.0   
Drama           289.0  7.957439  ...  1.164461e+08  924558264.0     241.0   
Family            2.0  7.800000  ...  3.273329e+08  435110554.0       2.0   
Fantasy           2.0  8.000000  ...  4.182577e+08  445151978.0       0.0   
Film-Noir         3.0  7.966667  ...  6.273068e+07  123353292.0       3.0   
Horror           11.0  7.909091  ...  1.362817e+08  298791505.0      11.0   
Mystery          12.0  7.975000  ...  1.310949e+08  474203697.0       8.0   
Thriller          1.0  7.800000  ...  1.755074e+07   17550741.0       1.0   
Western           4.0  8.350000  ...  1.920000e+07   31800000.0       4.0   

                                                                  
                mean        std   min    25%   50%    75%    max  
Genre                                                             
Action     73.419580  12.421252  33.0  65.00  74.0  82.00   98.0  
Adventure  78.437500  12.345393  41.0  69.75  80.5  87.25  100.0  
Animation  81.093333   8.813646  61.0  75.00  82.0  87.50   96.0  
Biography  76.240506  11.028187  48.0  70.50  76.0  84.50   97.0  
Comedy     78.720000  11.829160  45.0  72.00  79.0  88.00   99.0  
Crime      77.080460  13.099102  47.0  69.50  77.0  87.00  100.0  
Drama      79.701245  12.744687  28.0  72.00  82.0  89.00  100.0  
Family     79.000000  16.970563  67.0  73.00  79.0  85.00   91.0  
Fantasy          NaN        NaN   NaN    NaN   NaN    NaN    NaN  
Film-Noir  95.666667   1.527525  94.0  95.00  96.0  96.50   97.0  
Horror     80.000000  15.362291  46.0  77.50  87.0  88.50   97.0  
Mystery    79.125000  18.604435  52.0  65.25  77.0  98.50  100.0  
Thriller   81.000000        NaN  81.0  81.00  81.0  81.00   81.0  
Western    78.250000   9.032349  69.0  72.75  77.0  82.50   90.0  

[14 rows x 40 columns]

In [23]:
genres.sample(2,replace=True)

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
930,Watchmen,2009,162,Action,7.6,Zack Snyder,Jackie Earle Haley,500799,107509799.0,56.0
535,Dawn of the Dead,1978,127,Action,7.9,George A. Romero,David Emge,111512,5100000.0,71.0
329,The Martian,2015,144,Adventure,8.0,Ridley Scott,Matt Damon,760094,228433663.0,80.0
247,Lagaan: Once Upon a Time in India,2001,224,Adventure,8.1,Ashutosh Gowariker,Aamir Khan,105036,70147.0,84.0
920,The Secret of Kells,2009,71,Animation,7.6,Tomm Moore,Nora Twomey,31779,686383.0,81.0
906,Despicable Me,2010,95,Animation,7.6,Pierre Coffin,Chris Renaud,500851,251513985.0,72.0
38,The Pianist,2002,150,Biography,8.5,Roman Polanski,Adrien Brody,729603,32572577.0,85.0
952,The Hurricane,1999,146,Biography,7.6,Norman Jewison,Denzel Washington,91557,50668906.0,74.0
208,OMG: Oh My God!,2012,125,Comedy,8.1,Umesh Shukla,Paresh Rawal,51739,923221.0,NaN
541,Harold and Maude,1971,91,Comedy,7.9,Hal Ashby,Ruth Gordon,70826,804935438.0,62.0


In [24]:
genres.nunique()

,Series_Title,Released_Year,Runtime,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,
Action,172,61,78,15,123,121,172,172,50
Adventure,72,49,58,10,59,59,72,72,33
Animation,82,35,41,11,51,77,82,82,29
Biography,88,44,56,13,76,72,88,88,40
Comedy,155,72,70,11,113,133,155,155,44
Crime,106,56,65,14,86,85,107,107,39
Drama,289,83,95,14,211,250,288,287,52
Family,2,2,2,1,2,2,2,2,2
Fantasy,2,2,2,2,2,2,2,2,0


In [25]:
# agg methods
# passing dict
genres.agg(
    {
        'Runtime':'mean',
        'IMDB_Rating':'mean',
        'No_of_Votes':'sum',
        'Gross':'sum',
        'Metascore':'min'
    }
)

,Runtime,IMDB_Rating,No_of_Votes,Gross,Metascore
Genre,,,,,
Action,129.046512,7.949419,72282412,3.263226e+10,33.0
Adventure,134.111111,7.937500,22576163,9.496922e+09,41.0
Animation,99.585366,7.930488,21978630,1.463147e+10,61.0
Biography,136.022727,7.938636,24006844,8.276358e+09,48.0
Comedy,112.129032,7.901290,27620327,1.566387e+10,45.0
Crime,126.392523,8.016822,33533615,8.452632e+09,47.0
Drama,124.737024,7.957439,61367304,3.540997e+10,28.0
Family,107.500000,7.800000,551221,4.391106e+08,67.0
Fantasy,85.000000,8.000000,146222,7.827267e+08,NaN


In [26]:
# pasing list
genres.agg(['min' , 'max' , 'mean' , 'sum'])

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
# Adding both the syntax
genres.agg(
    {
        'Runtime':['min','mean'],
        'IMDB_Rating':'mean',
        'No_of_Votes':['sum','max'],
        'Gross':'sum',
        'Metascore':'min'
    }
)

Runtime             IMDB_Rating No_of_Votes                  Gross  \
              min        mean        mean         sum      max           sum   
Genre                                                                          
Action         45  129.046512    7.949419    72282412  2303232  3.263226e+10   
Adventure      88  134.111111    7.937500    22576163  1512360  9.496922e+09   
Animation      71   99.585366    7.930488    21978630   999790  1.463147e+10   
Biography      93  136.022727    7.938636    24006844  1213505  8.276358e+09   
Comedy         68  112.129032    7.901290    27620327   939631  1.566387e+10   
Crime          80  126.392523    8.016822    33533615  1826188  8.452632e+09   
Drama          64  124.737024    7.957439    61367304  2343110  3.540997e+10   
Family        100  107.500000    7.800000      551221   372490  4.391106e+08   
Fantasy        76   85.000000    8.000000      146222    88794  7.827267e+08   
Film-Noir     100  104.000000    7.966667      367215   158731  1.259105e+08   
Horror         71  102.090909    7.909091     3742556   787806  1.034649e+09   
Mystery        96  119.083333    7.975000     4203004  1129894  1.256417e+09   
Thriller      108  108.000000    7.800000       27733    27733  1.755074e+07   
Western       132  148.250000    8.350000     1289665   688390  5.822151e+07   

          Metascore  
                min  
Genre                
Action         33.0  
Adventure      41.0  
Animation      61.0  
Biography      48.0  
Comedy         45.0  
Crime          47.0  
Drama          28.0  
Family         67.0  
Fantasy         NaN  
Film-Noir      94.0  
Horror         46.0  
Mystery        52.0  
Thriller       81.0  
Western        69.0

In [30]:
# Create an empty list to store data pieces
frames = []

for group, data in genres:
    # Filter the highest rating in the group
    highest_in_group = data[data['IMDB_Rating'] == data['IMDB_Rating'].max()]
    frames.append(highest_in_group)

# Combine them all at once at the end
df = pd.concat(frames, ignore_index=True)
df

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
0,The Dark Knight,2008,152,Action,9.0,Christopher Nolan,Christian Bale,2303232,534858444.0,84.0
1,Interstellar,2014,169,Adventure,8.6,Christopher Nolan,Matthew McConaughey,1512360,188020017.0,74.0
2,Sen to Chihiro no kamikakushi,2001,125,Animation,8.6,Hayao Miyazaki,Daveigh Chase,651376,10055859.0,96.0
3,Schindler's List,1993,195,Biography,8.9,Steven Spielberg,Liam Neeson,1213505,96898818.0,94.0
4,Gisaengchung,2019,132,Comedy,8.6,Bong Joon Ho,Kang-ho Song,552778,53367844.0,96.0
5,La vita è bella,1997,116,Comedy,8.6,Roberto Benigni,Roberto Benigni,623629,57598247.0,59.0
6,The Godfather,1972,175,Crime,9.2,Francis Ford Coppola,Marlon Brando,1620367,134966411.0,100.0
7,The Shawshank Redemption,1994,142,Drama,9.3,Frank Darabont,Tim Robbins,2343110,28341469.0,80.0
8,E.T. the Extra-Terrestrial,1982,115,Family,7.8,Steven Spielberg,Henry Thomas,372490,435110554.0,91.0
9,Willy Wonka & the Chocolate Factory,1971,100,Family,7.8,Mel Stuart,Gene Wilder,178731,4000000.0,67.0


In [ ]:
genres.apply(min)

C:\Users\hites\AppData\Local\Temp\ipykernel_23888\1443917609.py:1: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
  genres.apply(min)
C:\Users\hites\AppData\Local\Temp\ipykernel_23888\1443917609.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genres.apply(min)


,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
Genre,,,,,,,,,,
Action,300,1924,45,Action,7.6,Abhishek Chaubey,Aamir Khan,25312,3296.0,NaN
Adventure,2001: A Space Odyssey,1925,88,Adventure,7.6,Akira Kurosawa,Aamir Khan,29999,61001.0,NaN
Animation,Akira,1940,71,Animation,7.6,Adam Elliot,Adrian Molina,25229,128985.0,NaN
Biography,12 Years a Slave,1928,93,Biography,7.6,Adam McKay,Adrien Brody,27254,21877.0,NaN
Comedy,(500) Days of Summer,1921,68,Comedy,7.6,Alejandro G. Iñárritu,Aamir Khan,26337,1305.0,NaN
Crime,12 Angry Men,1931,80,Crime,7.6,Akira Kurosawa,Ajay Devgn,27712,6013.0,NaN
Drama,1917,1925,64,Drama,7.6,Aamir Khan,Abhay Deol,25088,3600.0,NaN
Family,E.T. the Extra-Terrestrial,1971,100,Family,7.8,Mel Stuart,Gene Wilder,178731,4000000.0,67.0
Fantasy,Das Cabinet des Dr. Caligari,1920,76,Fantasy,7.9,F.W. Murnau,Max Schreck,57428,337574718.0,NaN


In [ ]:
# find no of movies starting with a for each group
movies.head(2)
def foo(group):
    return group['Series_Title'].str.startswith('A').sum()
genres.apply(foo)

C:\Users\hites\AppData\Local\Temp\ipykernel_23888\1562618544.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genres.apply(foo)


Genre
Action       10
Adventure     2
Animation     2
Biography     9
Comedy       14
Crime         4
Drama        21
Family        0
Fantasy       0
Film-Noir     0
Horror        1
Mystery       0
Thriller      0
Western       0
dtype: int64

In [ ]:
def rank_movie(group):
    group['genre_rank'] = group['IMDB_Rating'].rank(ascending=False)
    return group
genres.apply(rank_movie)

C:\Users\hites\AppData\Local\Temp\ipykernel_23888\2933735289.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genres.apply(rank_movie)


Series_Title Released_Year  \
Genre                                                                           
Action   2                                      The Dark Knight          2008   
         5        The Lord of the Rings: The Return of the King          2003   
         8                                            Inception          2010   
         10   The Lord of the Rings: The Fellowship of the Ring          2001   
         13               The Lord of the Rings: The Two Towers          2002   
...                                                         ...           ...   
Thriller 700                                    Wait Until Dark          1967   
Western  12                     Il buono, il brutto, il cattivo          1966   
         48                        Once Upon a Time in the West          1968   
         115                         Per qualche dollaro in più          1965   
         691                             The Outlaw Josey Wales          1976   

              Runtime     Genre  IMDB_Rating           Director  \
Genre                                                             
Action   2        152    Action          9.0  Christopher Nolan   
         5        201    Action          8.9      Peter Jackson   
         8        148    Action          8.8  Christopher Nolan   
         10       178    Action          8.8      Peter Jackson   
         13       179    Action          8.7      Peter Jackson   
...               ...       ...          ...                ...   
Thriller 700      108  Thriller          7.8      Terence Young   
Western  12       161   Western          8.8       Sergio Leone   
         48       165   Western          8.5       Sergio Leone   
         115      132   Western          8.3       Sergio Leone   
         691      135   Western          7.8     Clint Eastwood   

                          Star1  No_of_Votes        Gross  Metascore  \
Genre                                                                  
Action   2       Christian Bale      2303232  534858444.0       84.0   
         5          Elijah Wood      1642758  377845905.0       94.0   
         8    Leonardo DiCaprio      2067042  292576195.0       74.0   
         10         Elijah Wood      1661481  315544750.0       92.0   
         13         Elijah Wood      1485555  342551365.0       87.0   
...                         ...          ...          ...        ...   
Thriller 700     Audrey Hepburn        27733   17550741.0       81.0   
Western  12      Clint Eastwood       688390    6100000.0       90.0   
         48         Henry Fonda       302844    5321508.0       80.0   
         115     Clint Eastwood       232772   15000000.0       74.0   
         691     Clint Eastwood        65659   31800000.0       69.0   

              genre_rank  
Genre                     
Action   2           1.0  
         5           2.0  
         8           3.5  
         10          3.5  
         13          6.0  
...                  ...  
Thriller 700         1.0  
Western  12          1.0  
         48          2.0  
         115         3.0  
         691         4.0  

[1000 rows x 11 columns]

In [ ]:
# find normalized IMDB rating group wise

def normal(group):
  group['norm_rating'] = (group['IMDB_Rating'] - group['IMDB_Rating'].min())/(group['IMDB_Rating'].max() - group['IMDB_Rating'].min())
  return group

genres.apply(normal)

C:\Users\hites\AppData\Local\Temp\ipykernel_23888\2908049479.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  genres.apply(normal)


Series_Title Released_Year  \
Genre                                                                           
Action   2                                      The Dark Knight          2008   
         5        The Lord of the Rings: The Return of the King          2003   
         8                                            Inception          2010   
         10   The Lord of the Rings: The Fellowship of the Ring          2001   
         13               The Lord of the Rings: The Two Towers          2002   
...                                                         ...           ...   
Thriller 700                                    Wait Until Dark          1967   
Western  12                     Il buono, il brutto, il cattivo          1966   
         48                        Once Upon a Time in the West          1968   
         115                         Per qualche dollaro in più          1965   
         691                             The Outlaw Josey Wales          1976   

              Runtime     Genre  IMDB_Rating           Director  \
Genre                                                             
Action   2        152    Action          9.0  Christopher Nolan   
         5        201    Action          8.9      Peter Jackson   
         8        148    Action          8.8  Christopher Nolan   
         10       178    Action          8.8      Peter Jackson   
         13       179    Action          8.7      Peter Jackson   
...               ...       ...          ...                ...   
Thriller 700      108  Thriller          7.8      Terence Young   
Western  12       161   Western          8.8       Sergio Leone   
         48       165   Western          8.5       Sergio Leone   
         115      132   Western          8.3       Sergio Leone   
         691      135   Western          7.8     Clint Eastwood   

                          Star1  No_of_Votes        Gross  Metascore  \
Genre                                                                  
Action   2       Christian Bale      2303232  534858444.0       84.0   
         5          Elijah Wood      1642758  377845905.0       94.0   
         8    Leonardo DiCaprio      2067042  292576195.0       74.0   
         10         Elijah Wood      1661481  315544750.0       92.0   
         13         Elijah Wood      1485555  342551365.0       87.0   
...                         ...          ...          ...        ...   
Thriller 700     Audrey Hepburn        27733   17550741.0       81.0   
Western  12      Clint Eastwood       688390    6100000.0       90.0   
         48         Henry Fonda       302844    5321508.0       80.0   
         115     Clint Eastwood       232772   15000000.0       74.0   
         691     Clint Eastwood        65659   31800000.0       69.0   

              norm_rating  
Genre                      
Action   2       1.000000  
         5       0.928571  
         8       0.857143  
         10      0.857143  
         13      0.785714  
...                   ...  
Thriller 700          NaN  
Western  12      1.000000  
         48      0.700000  
         115     0.500000  
         691     0.000000  

[1000 rows x 11 columns]

In [ ]:
# groupby on multiple cols
duo = movies.groupby(['Director','Star1'])
duo
# size
duo.size()
# get_group
duo.get_group(('Aamir Khan','Amole Gupte'))

,Series_Title,Released_Year,Runtime,Genre,IMDB_Rating,Director,Star1,No_of_Votes,Gross,Metascore
65,Taare Zameen Par,2007,165,Drama,8.4,Aamir Khan,Amole Gupte,168895,1223869.0,NaN


In [ ]:
# find the most earning actor->director combo
duo['Gross'].sum().sort_values(ascending=False).head(1)

Director        Star1         
Akira Kurosawa  Toshirô Mifune    2.999877e+09
Name: Gross, dtype: float64

In [ ]:
# find the best(in-terms of metascore(avg)) actor->genre combo
movies.groupby(['Star1','Genre'])['Metascore'].mean().reset_index().sort_values('Metascore',ascending=False).head(1)

,Star1,Genre,Metascore
606,Peter O'Toole,Adventure,100.0


In [ ]:
# agg on multiple groupby
duo.agg(['min','max','mean'])

TypeError: agg function failed [how->mean,dtype->object]